In [204]:
import pandas as pd
import numpy as np
import spacy
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

nlp = spacy.load("en_core_web_sm")

df = pd.read_csv("datathon_year_3.csv")

# rows, cols
print(df.shape)
# types
print(df.dtypes)

(3995, 15)
Idea #                                int64
Original Title                          str
Challenge                               str
Solution                                str
Audience                                str
Borough                                 str
Neighborhood(s)                         str
Host of Idea Generation Workshop        str
Impact Area                             str
Sub-Category                            str
Type of Idea                            str
Idea Zip Code                       float64
Idea Generation Session Zip Code    float64
Idea Source                             str
Insights                                str
dtype: object


In [205]:
# preview of first 2 rows
print(df.head(2))

   Idea #                          Original Title  \
0      14  Pathways to Success: Bilingual Support   
1     111                    Youth Launch Project   

                                           Challenge  \
0  Teens of immigrants struggle with the college ...   
1  A challenge in Harlem is helping students lear...   

                                            Solution    Audience    Borough  \
0  A mentorship program that would offer bilingua...  Immigrants     Queens   
1  A youth enterprenuership program that teaches ...       Youth  Manhattan   

  Neighborhood(s)                   Host of Idea Generation Workshop  \
0             NaN  [10/10/2024 @ 2:00 PM] DOE Civics For All Even...   
1          Harlem  [10/15/2024 @ 5:30 PM] Lee Heritage Media LLC ...   

  Impact Area                                       Sub-Category  \
0   Education  EDU - Adult Educational Programs,EDU - Aftersc...   
1   Education  EDU - Professional Skills Development,EDU - Jo...   

           

In [206]:
# drop unnecessary columns 
df = df.drop(['Host of Idea Generation Workshop', 'Idea Zip Code', 'Idea Generation Session Zip Code', 'Idea Source'], axis = 1)
print(df.dtypes)

Idea #             int64
Original Title       str
Challenge            str
Solution             str
Audience             str
Borough              str
Neighborhood(s)      str
Impact Area          str
Sub-Category         str
Type of Idea         str
Insights             str
dtype: object


In [207]:
# check for NA values
print(df.isna().sum())

Idea #               0
Original Title      41
Challenge            6
Solution            13
Audience             1
Borough              0
Neighborhood(s)    201
Impact Area          4
Sub-Category       474
Type of Idea       703
Insights           838
dtype: int64


In [208]:
# Check rows where Challenge or Solution is duplicated
duplicated_rows = df[df.duplicated(subset=["Challenge"], keep=False) |
                     df.duplicated(subset=["Solution"], keep=False)]

# preview of dupes
print("Duplicated rows:\n", duplicated_rows)

# drop duplicates
df = df.drop_duplicates(subset=["Challenge"], keep='first')
df = df.drop_duplicates(subset=["Solution"], keep='first')

Duplicated rows:
       Idea #                                     Original Title  \
112      680            Housing Support for the Senior Citizens   
196     1580                          Disability Center For All   
239     1977                          Teaching kids STEM skills   
240     1980        A program to teach them community building    
533      470                                      In The Loope!   
...      ...                                                ...   
3909    4405  Bridging Tomorrow: FREE Generative Learning AI...   
3952    4464  Bridging the Exponentially Growing Digital Div...   
3958    4551  Bridging Tomorrow: FREE Generative Learning AI...   
3962    4332  "We Grow Life" is a citywide initiative to cul...   
3976    4499  "We Grow Life" is a citywide initiative to cul...   

                                              Challenge  \
112   It's hard to find and navigate public housing ...   
196   Immigrant families lack assistance for family ...   


In [209]:
# NaN Check
# If NaN left in data, it changes the column to a float causing errors
df["Challenge"].apply(type).value_counts()

# drop missing values
df = df.dropna(subset=["Challenge", "Solution"])

In [210]:
# remove characters from rows that start with bad chars
df[['Challenge', 'Solution']] = df[['Challenge', 'Solution']].apply(
    lambda col: col.str.lstrip("\"'@-=")
)

# ^^^ above uses anon func lambda to apply the following func into each col, acts as wrapper so can use .apply
# equivalent to:
# def clean_start(col):
#     return col.str.lstrip("\"'@-=")

# df[cols_to_clean] = df[cols_to_clean].apply(clean_start)

# remove "=" from text
# for each selected column, replace characters in all strings
df[['Challenge', 'Solution']] = df[['Challenge', 'Solution']].apply(
    lambda col: col.str.replace("=", "", regex=False))

In [211]:
# noun chunk extraction
# removes pronouns and stop words from results
# also remove custom stop phrases that were common
STOP_PHRASES = {"people", "nyc", "the bronx", "a lot", "lack", "community", "bronx", "need", "help", "access", "lot"}

def extract_noun_chunk(text):
    doc = nlp(text.lower())
    chunks = []
    for chunk in doc.noun_chunks:
        # skip pronouns and root stop words
        if chunk.root.pos_ != "PRON" and not chunk.root.is_stop:
            # remove leading stop words
            cleaned = " ".join([token.text for token in chunk if not token.is_stop])
            # skip chunks in stop list
            if cleaned and cleaned not in STOP_PHRASES:
                chunks.append(cleaned)
    return " | ".join(chunks)

df['Challenge_noun_chunk'] = df['Challenge'].apply(extract_noun_chunk)
df['Solution_noun_chunk'] = df['Solution'].apply(extract_noun_chunk)
temp_df_nc = df[['Challenge_noun_chunk', 'Solution_noun_chunk']]
print(temp_df_nc.head(10))

                                Challenge_noun_chunk  \
0  teens | immigrants | college application proce...   
1  challenge | harlem | students | business finan...   
2  seniors | limited credit | income | housing | ...   
3                teens | methods | income | canarsie   
4                younger parents | parents | parents   
5  schools | rigid structure | time | extra curri...   
6  clubs | elementary school grades | high school...   
7  pakistani women immigrants | support form part...   
8  “ relevant ” work opportunities | teens / yout...   
9                                                      

                                 Solution_noun_chunk  
0  mentorship program | bilingual workshops | imm...  
1  youth enterprenuership program | students | bu...  
2  urgent need | senior apartments | queens | hou...  
3  program | teens | available resources | employ...  
4  parenting coaching classes | young adults | su...  
5  programs | host clubs | students | knowledge |... 

In [212]:
# most common issues from noun chunk Challenge
all_chunks = []
for chunks in df["Challenge_noun_chunk"]:
    all_chunks.extend(chunks.split(" | "))

Counter(all_chunks).most_common(100)

[('youth', 420),
 ('children', 231),
 ('immigrants', 187),
 ('parents', 187),
 ('school', 181),
 ('kids', 174),
 ('resources', 168),
 ('seniors', 151),
 ('programs', 150),
 ('young people', 144),
 ('streets', 142),
 ('disabilities', 140),
 ('housing', 139),
 ('problem', 135),
 ('money', 130),
 ('students', 118),
 ('older adults', 118),
 ('jobs', 118),
 ('education', 110),
 ('english', 109),
 ('support', 103),
 ('veterans', 102),
 ('food', 101),
 ('schools', 98),
 ('challenge', 95),
 ('life', 76),
 ('families', 73),
 ('mental health', 73),
 ('adults', 71),
 ('place', 70),
 ('drugs', 69),
 ('job', 69),
 ('services', 68),
 ('time', 67),
 ('communities', 67),
 ('challenges', 65),
 ('opportunities', 63),
 ('city', 63),
 ('homeless people', 63),
 ('things', 62),
 ('neighborhood', 60),
 ('activities', 59),
 ('residents', 59),
 ('individuals', 57),
 ('technology', 56),
 ('issue', 55),
 ('public housing', 55),
 ('problems', 55),
 ('safety', 54),
 ('program', 54),
 ('work', 53),
 ('knowledge', 5

In [213]:
# group noun chunks by borough
def split_chunks(text):
    return text.split(" | ")

df["chunk_list"] = df["Challenge_noun_chunk"].apply(split_chunks)

exploded = df.explode("chunk_list")

borough_counts = (
    exploded.groupby("Borough")["chunk_list"]
    .value_counts()
)

for borough, group in borough_counts.groupby(level=0):
    print(f"\nTop challenges in {borough}:")
    print(group.head(25))


Top challenges in Bronx:
Borough  chunk_list  
Bronx    youth           114
         children         64
         school           55
         resources        51
         young people     47
         programs         47
         kids             47
         veterans         47
         immigrants       46
         streets          40
         money            40
         parents          40
         problem          38
         education        38
         students         38
         housing          37
         support          35
         food             34
         disabilities     34
         schools          33
         services         33
         seniors          31
         challenge        31
         life             30
         jobs             28
Name: count, dtype: int64

Top challenges in Brooklyn:
Borough   chunk_list     
Brooklyn  youth              95
          immigrants         61
          children           43
          resources          43
          seniors 

In [ ]:
# Jaccard Similarity: Comparison of similarity between borough results
# Treat each borough’s top noun chunks as a set
# then measures overlap between two boroughs top issues

# Get top noun chunks for each borough
borough_top_chunks = {
    borough: set(group.head(30).index.get_level_values(1))
    for borough, group in borough_counts.groupby(level=0)
}

boroughs = list(borough_top_chunks.keys())

jaccard_df = pd.DataFrame(index=boroughs, columns=boroughs, dtype=float)

# Compute Jaccard similarity for each pair
for b1 in boroughs:
    for b2 in boroughs:
        set1 = borough_top_chunks[b1]
        set2 = borough_top_chunks[b2]
        jaccard_df.loc[b1, b2] = len(set1 & set2) / len(set1 | set2)

# round for readability
jaccard_df = jaccard_df.round(2)

print(jaccard_df)

Jaccard similarity: 0.7142857142857143


In [178]:
# lemmatize function
def clean_lemmatize(text):
    doc = nlp(text.lower())
    return " ".join([
        token.lemma_
        for token in doc
        # punctuation   
        if not token.is_punct   
        # spaces 
        and not token.is_space   
        # stopwords
        and not token.is_stop  
    ])

df['Challenge_lem'] = df['Challenge'].apply(clean_lemmatize)
df['Solution_lem'] = df['Solution'].apply(clean_lemmatize)
lem_df = df[['Challenge_lem', 'Solution_lem']]
print(lem_df.head(20))

                                        Challenge_lem  \
0   teen immigrant struggle college application pr...   
1   challenge harlem help student learn start busi...   
2   senior limited credit income find housing majo...   
3                         teen method income canarsie   
4                young parent parent help grow parent   
5   school rigid structure leave time extra curric...   
6   club elementary school grade like high school ...   
7   pakistani woman immigrant unemployed support f...   
8   lack relevant work opportunity teen youth word...   
9                              people drown know swim   
10  lack access free financial literacy education ...   
11  new york city housing crisis landlord advantag...   
12                youth outlet express gift misdirect   
13                              people cook mean feed   
14  folk limited access housing struggle food inse...   
15  youth employment trade programming outreach of...   
16  awareness autism student di

In [179]:
# dependency parsing

# def show_dependencies(text):
#     doc = nlp(text)
#     for token in doc:
#         print(token.text, token.dep_, token.head.text)

# subject object verb extraction, including prepositional objs
def extract_svo(text):
    doc = nlp(text)
    svos = []
    
    for token in doc:
        if token.pos_ == "VERB":
            subjects = [child for child in token.children if child.dep_ in ["nsubj", "nsubjpass"]]
            objects = [child for child in token.children if child.dep_ in ["dobj", "attr", "oprd"]]
            
            # include prepositional objects
            for prep in [child for child in token.children if child.dep_ == "prep"]:
                objects.extend([child for child in prep.children if child.dep_ == "pobj"])
            
            for subj in subjects:
                for obj in objects:
                    svos.append((subj.text, token.lemma_, obj.text))
    
    return svos

df["svos_problems"] = df["Challenge"].apply(extract_svo)
df["svos_solutions"] = df["Solution"].apply(extract_svo)

svo_df = df[['svos_problems', 'svos_solutions']]
print(svo_df.head(20))

# test for checking visualization of SVO on row 5 (cant use on whole column)
# for text in df["Community problems"].head(5):
#     doc = nlp(text)
#     displacy.render(doc, style="dep", jupyter=True)

                                        svos_problems  \
0   [(Teens, struggle, processes), (parents, advis...   
1                                                  []   
2                                                  []   
3                            [(teens, have, methods)]   
4                          [(parents, grow, parents)]   
5   [(Schools, have, structure), (that, leave, tim...   
6   [(This, offer, experiences), (This, offer, age...   
7                                                  []   
8                                                  []   
9                                                  []   
10                                                 []   
11  [(that, take, advantage), (that, take, people)...   
12                           [(youth, have, outlets)]   
13                                                 []   
14                                                 []   
15              [(employment, oftentime, experience)]   
16             [(child, have, I

In [ ]:
# Most common issues from challenge lem
# shows what "issues" are most common among rows
all_words = " ".join(df["Challenge_lem"]).split()
word_counts = Counter(all_words).most_common(50)
df_counts = pd.DataFrame(word_counts, columns=["Common Issue", "Count"])
print(df_counts)

   Common Issue  Count
0        people   1452
1     community    960
2          need    834
3         youth    629
4          lack    588
5       program    539
6        school    529
7          help    516
8       housing    463
9           job    350
10        child    341
11       health    332
12    immigrant    322
13        adult    295
14        young    292
15       mental    291
16    challenge    290
17      problem    285
18          low    285
19      support    281
20         food    271
21       senior    269
22       parent    267
23       income    267
24     resource    267
25       access    266
26      english    266
27       street    265
28         like    251
29       public    248
30          old    242
31         work    239
32      service    237
33        issue    235
34  opportunity    231
35          lot    230
36       family    226
37          new    225
38    education    223
39          kid    215
40         high    205
41   disability    199
42     acti

In [182]:
# SVO relationship analysis
# singular verbs freq
verbs = []
for row in df["svos_problems"]:
    for (_, verb, _) in row:
        verbs.append(verb)

Counter(verbs).most_common(50)

[('have', 708),
 ('need', 419),
 ('see', 197),
 ('get', 138),
 ('face', 127),
 ('go', 77),
 ('take', 72),
 ('live', 67),
 ('lack', 65),
 ('use', 63),
 ('provide', 58),
 ('speak', 58),
 ('leave', 54),
 ('help', 54),
 ('lead', 52),
 ('afford', 48),
 ('do', 48),
 ('come', 46),
 ('know', 46),
 ('offer', 44),
 ('struggle', 41),
 ('impact', 41),
 ('give', 40),
 ('experience', 39),
 ('affect', 38),
 ('make', 37),
 ('find', 36),
 ('create', 33),
 ('spend', 31),
 ('learn', 29),
 ('suffer', 28),
 ('include', 27),
 ('receive', 25),
 ('become', 25),
 ('feel', 24),
 ('keep', 24),
 ('understand', 23),
 ('teach', 23),
 ('like', 22),
 ('pay', 22),
 ('support', 20),
 ('limit', 20),
 ('benefit', 20),
 ('work', 20),
 ('lose', 19),
 ('put', 19),
 ('address', 19),
 ('cause', 18),
 ('deserve', 16),
 ('engage', 16)]

In [183]:
# Subject–verb, verb–object 
pairs = []
for row in df["svos_problems"]:
    for (subj, verb, obj) in row:
        pairs.append((verb, obj))

Counter(pairs).most_common(50)

[(('need', 'help'), 59),
 (('see', 'community'), 54),
 (('have', 'access'), 41),
 (('have', 'time'), 35),
 (('see', 'that'), 31),
 (('speak', 'English'), 28),
 (('face', 'challenges'), 22),
 (('need', 'programs'), 20),
 (('need', 'support'), 18),
 (('have', 'resources'), 17),
 (('have', 'opportunity'), 16),
 (('have', 'programs'), 15),
 (('have', 'place'), 15),
 (('get', 'help'), 15),
 (('have', 'experience'), 13),
 (('need', 'assistance'), 12),
 (('have', 'lot'), 12),
 (('have', 'money'), 12),
 (('spend', 'time'), 11),
 (('see', 'people'), 11),
 (('have', 'trouble'), 11),
 (('need', 'activities'), 11),
 (('see', 'lot'), 11),
 (('have', 'space'), 10),
 (('have', 'opportunities'), 10),
 (('have', 'problem'), 10),
 (('lack', 'access'), 9),
 (('impact', 'people'), 9),
 (('have', 'support'), 9),
 (('have', 'difficulty'), 9),
 (('have', 'knowledge'), 9),
 (('face', 'barriers'), 9),
 (('take', 'care'), 8),
 (('have', 'skills'), 8),
 (('need', 'space'), 8),
 (('have', 'issues'), 8),
 (('need'

In [184]:
# problem-to-solution column mapping
mapping = []

for i, row in df.iterrows():
    for (subj, verb, obj) in row["svos_problems"]:
        for (s_subj, s_verb, s_obj) in row["svos_solutions"]:
            mapping.append((verb + " " + obj, s_verb + " " + s_obj))

Counter(mapping).most_common(50)

[(('leave terms', 'offer programs'), 18),
 (('leave dust', 'offer programs'), 18),
 (('stem challenges', 'offer programs'), 18),
 (('lack training', 'offer programs'), 13),
 (('retain character', 'secure employment'), 12),
 (('miss activities', 'offer programs'), 10),
 (('retain character', 'train residents'), 9),
 (('advance speeds', 'offer programs'), 9),
 (('release functionalities', 'offer programs'), 9),
 (('respond city', 'offer programs'), 9),
 (('experience disparities', 'offer programs'), 9),
 (('take that', 'offer programs'), 9),
 (('live lack', 'offer programs'), 9),
 (('live inequities', 'offer programs'), 9),
 (('have quality', 'offer programs'), 9),
 (('exist communities', 'offer programs'), 9),
 (('widen pace', 'offer programs'), 9),
 (('acquire age', 'offer programs'), 9),
 (('lose time', 'offer programs'), 9),
 (('exist NYC', 'offer programs'), 9),
 (('lack access', 'offer programs'), 9),
 (('make strides', 'offer programs'), 9),
 (('accomplish what', 'offer programs')

In [ ]:
# Term Frequency-Inverse Document Frequency (TF-IDF)
# TF-IDF score is how important a word or phrase is across data
# Score high when term frequent in some responses, but is not everywhere
# AKA not "most common", "most severe", etc
vectorizer = TfidfVectorizer(
    stop_words='english',
    token_pattern=r'\b[a-zA-Z]{3,}\b',
    min_df=3,
    max_df=0.8,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df['Challenge_lem'])

feature_names = vectorizer.get_feature_names_out()

# compute average TF-IDF scores
scores = np.asarray(X.mean(axis=0)).ravel()

# get top 30 terms
top_indices = scores.argsort()[::-1][:30]

top_terms_df = pd.DataFrame({
    "term": [feature_names[i] for i in top_indices],
    "score": [round(float(scores[i]), 4) for i in top_indices]
})
print(top_terms_df)

         term   score
0      people  0.0394
1   community  0.0272
2        need  0.0257
3       youth  0.0246
4     program  0.0205
5        lack  0.0204
6     housing  0.0195
7        help  0.0193
8      school  0.0187
9         job  0.0146
10  immigrant  0.0144
11      child  0.0134
12     senior  0.0134
13     health  0.0133
14      adult  0.0131
15    problem  0.0127
16     mental  0.0126
17       food  0.0125
18    english  0.0122
19      young  0.0119
20     street  0.0119
21        old  0.0117
22     parent  0.0117
23     income  0.0116
24   resource  0.0115
25     public  0.0113
26        low  0.0112
27        lot  0.0108
28   homeless  0.0107
29  challenge  0.0105


In [ ]:
# clustering
# outputs themes of community health challenges
kmeans = KMeans(n_clusters=5)
# X is TF-IDF matrix from prev
kmeans.fit(X)

df["cluster"] = kmeans.labels_

terms = vectorizer.get_feature_names_out()

for i in range(5):
    center = kmeans.cluster_centers_[i]
    top_terms = [terms[j] for j in center.argsort()[::-1][:10]]
    print(f"Cluster {i}: {top_terms}")

Cluster 0: ['community', 'lack', 'health', 'mental', 'mental health', 'youth', 'resource', 'challenge', 'service', 'need']
Cluster 1: ['people', 'young people', 'disability', 'young', 'homeless', 'help', 'people disability', 'need', 'homeless people', 'lot']
Cluster 2: ['housing', 'income', 'low', 'low income', 'public housing', 'public', 'affordable', 'people', 'income people', 'affordable housing']
Cluster 3: ['program', 'adult', 'old', 'old adult', 'youth', 'school', 'school program', 'program youth', 'need', 'help']
Cluster 4: ['school', 'youth', 'need', 'immigrant', 'child', 'senior', 'parent', 'job', 'help', 'food']


In [187]:
# Network analysis (SVO challenges)
# maps extracted entities and relationships into graph of nodes (concepts) and edges (connections)
# helps identify key issues and how different problems or groups are interconnected 
edges = []

for row in df["svos_problems"]:
    for (subj, verb, obj) in row:
        edges.append((subj, f"{verb} {obj}"))

edge_counts = Counter(edges)

edge_df = pd.DataFrame(edge_counts.items(), columns=["connection", "count"])
edge_df[["subject", "object"]] = pd.DataFrame(edge_df["connection"].tolist(), index=edge_df.index)

edge_df = edge_df.sort_values(by="count", ascending=False)

print(edge_df.head(20))

                    connection  count  subject          object
68          (I, see community)     40        I   see community
556              (I, see that)     26        I        see that
331       (who, speak English)     12      who   speak English
650               (I, see lot)     11        I         see lot
254        (we, see community)     11       we   see community
70       (people, have access)      9   people     have access
1348       (We, need programs)      9       We   need programs
793        (people, need help)      8   people       need help
657            (I, see people)      8        I      see people
57      (they, have resources)      6     they  have resources
1905         (they, need help)      6     they       need help
919   (schools, offer courses)      6  schools   offer courses
160          (they, need what)      5     they       need what
276        (people, have time)      5   people       have time
2719       (People, need help)      5   People       ne

In [ ]:
# Network analysis on solutions
edges = []

for row in df["svos_solutions"]:
    for (subj, verb, obj) in row:
        edges.append((subj, f"{verb} {obj}"))

edge_counts = Counter(edges)

edge_df_sol = pd.DataFrame(edge_counts.items(), columns=["connection", "count"])
edge_df_sol[["subject", "object"]] = pd.DataFrame(edge_df_sol["connection"].tolist(), index=edge_df_sol.index)

edge_df_sol = edge_df_sol.sort_values(by="count", ascending=False)

print(edge_df_sol.head(20))

                        connection  count   subject                object
712              (that, connect _)     12      that             connect _
186            (that, help people)     10      that           help people
36             (We, solve problem)      9        We         solve problem
200              (they, need what)      9      they             need what
137             (We, need program)      8        We          need program
906                 (who, need it)      8       who               need it
124       (that, provide services)      6      that      provide services
844              (that, help them)      6      that             help them
151        (that, provide support)      6      that       provide support
190             (that, take place)      6      that            take place
333                (they, do what)      5      they               do what
295          (who, speak language)      5       who        speak language
934               (who, need help)    

In [189]:
# output
# in-notebook display
df

# csv export
df.to_csv("output.csv", index=False)